## CELL 00 — Binary Left-vs-Right Motor-Imagery EEG Pipeline

This notebook is the binary version of the supplied S³ Spectral-Spatial Domain Adaptation pipeline.

**Task:** subject-independent classification of **Left-Hand MI vs Right-Hand MI** using PhysioNet EEGMMIDB / EEG Motor Movement-Imagery runs **4, 8, 12** only.

**Architecture:** per-trial channel Z-score → learnable Sinc filter bank → dynamic GNN → bidirectional GRU (the supplied `SimplifiedBiMamba` implementation) → SE attention → classification + GRL domain alignment + supervised contrastive learning → AdaBN target-statistics adaptation.

**Important:** this code uses a bidirectional GRU, not a true Mamba state-space model. The latent diffusion module described in the reference manuscript is not implemented.


## CELL 01 — Imports and environment

This cell imports the numerical, EEG, deep-learning, and evaluation libraries.

The important packages are:
- **MNE** for reading PhysioNet EEGMMIDB EDF files, annotations, and epoching.
- **PyTorch** for the model and training.
- **scikit-learn** for accuracy, Cohen's kappa, classification metrics, ROC/AUC, and t-SNE.
- **pandas / NumPy / Matplotlib** for experiment logging and figures.

In [1]:
# ============================================================
# CELL 01 — Imports and environment
# ============================================================

from pathlib import Path
import os, math, json, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score,
    confusion_matrix, classification_report, roc_curve, auc,
    roc_auc_score
)
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

print("Imports loaded successfully.")


Imports loaded successfully.


## CELL 02 — Configuration and reproducibility

This cell defines the experimental settings.

The source configuration uses:
- 109 possible subjects
- runs 4, 6, 8, 10, 12, and 14
- 250 Hz sampling
- 4-second trials
- 22 EEG channels
- 4 classes
- batch size 64
- 100 training epochs
- AdamW with learning rate `1e-3`
- cosine annealing
- label smoothing, domain loss, and supervised contrastive loss

A fixed random seed of 42 is used. The code automatically selects CUDA, then Apple MPS, then CPU. fileciteturn1file0L76-L137

In [7]:
# ============================================================
# CELL 02 — DEVICE CONFIGURATION
# ============================================================

import torch

# Automatically use GPU when available
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("=" * 70)
print("DEVICE CONFIGURATION")
print("=" * 70)

print("PyTorch version :", torch.__version__)
print("Device          :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU             :", torch.cuda.get_device_name(0))
    print(
        "CUDA memory     :",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

elif DEVICE.type == "mps":
    print("Apple Silicon GPU: MPS")

else:
    print("Using CPU")

print("=" * 70)
# ============================================================
# CELL 02 — BINARY LEFT-vs-RIGHT OPTIMIZED CONFIGURATION
# ============================================================

SEED = 42

# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------

DATA_DIR = "./eegmmidb"

# Only Left/Right motor imagery runs
RUNS = [4, 8, 12]

TMIN = 0.0
TMAX = 4.0
FS = 250.0

N_CHANNELS = 22

# ------------------------------------------------------------
# BINARY CLASSES
# ------------------------------------------------------------

N_CLASSES = 2

CLASS_NAMES = [
    "Left Hand MI",
    "Right Hand MI"
]

# ------------------------------------------------------------
# EXACT TEST SUBJECTS REQUESTED
# ------------------------------------------------------------

TEST_SUBJECTS = [
    4,
    15,
    23,
    29,
    31,
    42,
    55,
    71,
    82,
    95
]

NUM_TEST_FOLDS = len(TEST_SUBJECTS)

# ------------------------------------------------------------
# SOURCE TRAINING SUBJECTS
# ------------------------------------------------------------

TOTAL_SUBJECTS = 109

# All available source subjects except the target
USE_ALL_SOURCE_SUBJECTS = True

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

BATCH_SIZE = 64

NUM_EPOCHS = 100

LR = 3e-4

MIN_LR = 1e-6

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.03

GRAD_CLIP = 1.0

# ------------------------------------------------------------
# DOMAIN ADVERSARIAL LEARNING
# ------------------------------------------------------------
#
# Reduced so binary classification dominates the objective.
# ------------------------------------------------------------

DOMAIN_WEIGHT = 0.10

# ------------------------------------------------------------
# SUPERVISED CONTRASTIVE LEARNING
# ------------------------------------------------------------

SUPCON_WEIGHT = 0.10

SUPCON_TEMP = 0.10

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

NUM_FILTERS = 10

SINC_KERNEL = 81

SPATIAL_DIM = 64

# ------------------------------------------------------------
# TEMPORAL COMPRESSION
# ------------------------------------------------------------

TEMPORAL_DOWNSAMPLE = 4

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

RESULTS_DIR = Path(
    "./results_binary_exact10_optimized"
)

FIG_DIR = (
    RESULTS_DIR /
    "figures"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DOMAIN_CLASSES = TOTAL_SUBJECTS

print("=" * 80)
print("BINARY LEFT-vs-RIGHT OPTIMIZED CONFIGURATION")
print("=" * 80)

print("Classes:")
print("  0 =", CLASS_NAMES[0])
print("  1 =", CLASS_NAMES[1])

print("\nRuns:", RUNS)

print(
    "\nExact test subjects:",
    [
        f"S{s:03d}"
        for s in TEST_SUBJECTS
    ]
)

print("\nTraining configuration:")
print("  Batch size       :", BATCH_SIZE)
print("  Epochs           :", NUM_EPOCHS)
print("  Learning rate    :", LR)
print("  Weight decay     :", WEIGHT_DECAY)
print("  Domain weight    :", DOMAIN_WEIGHT)
print("  SupCon weight    :", SUPCON_WEIGHT)
print("  Temporal factor  :", TEMPORAL_DOWNSAMPLE)

print("\n[OK] Configuration loaded.")

DEVICE CONFIGURATION
PyTorch version : 2.10.0
Device          : mps
Apple Silicon GPU: MPS
BINARY LEFT-vs-RIGHT OPTIMIZED CONFIGURATION
Classes:
  0 = Left Hand MI
  1 = Right Hand MI

Runs: [4, 8, 12]

Exact test subjects: ['S004', 'S015', 'S023', 'S029', 'S031', 'S042', 'S055', 'S071', 'S082', 'S095']

Training configuration:
  Batch size       : 64
  Epochs           : 100
  Learning rate    : 0.0003
  Weight decay     : 0.0001
  Domain weight    : 0.1
  SupCon weight    : 0.1
  Temporal factor  : 4

[OK] Configuration loaded.


## CELL 03 — Dataset loader

`EEGMMIDB_Dataset` scans subject folders (`S001`, `S002`, …), loads the selected EDF runs, keeps the first 22 channels, resamples to 250 Hz, reads annotation events, and creates 4-second epochs.

The run-dependent label mapping is:

| Runs | T1 | T2 |
|---|---|---|
| 4, 8, 12 | Left Fist = 0 | Right Fist = 1 |
| 6, 10, 14 | Both Fists = 2 | Both Feet = 3 |

The dataset also stores a **subject ID** for the domain-adaptation loss and the run ID for traceability. Each trial is then normalized independently, channel by channel, using its own temporal mean and standard deviation. fileciteturn1file0L143-L240

In [8]:
# ============================================================
# CELL 03 — Binary EEGMMIDB dataset loader
# ============================================================

class EEGMMIDB_Dataset(Dataset):
    """
    Binary EEGMMIDB loader for Left-Hand MI vs Right-Hand MI.

    Only runs 4, 8 and 12 are used. In these runs:
        T1 -> Left Hand MI
        T2 -> Right Hand MI

    MNE's events_from_annotations() returns the ACTUAL numeric
    event IDs. Those IDs must be used for MNE Epochs. We then
    convert T1/T2 into our ML labels 0/1.
    """

    def __init__(self, data_dir, subjects, runs=RUNS, tmin=TMIN, tmax=TMAX):
        self.data_dir = str(data_dir)
        self.subjects = list(subjects)
        self.runs = list(runs)
        self.tmin = tmin
        self.tmax = tmax

        self.epochs = []
        self.labels = []
        self.subject_ids = []
        self.run_ids = []

        self.load_data()

    @staticmethod
    def _find_event_code(event_id_dict, target_name):
        target_name = str(target_name).strip().upper()

        for description, code in event_id_dict.items():
            desc = str(description).strip().upper()

            if desc == target_name:
                return int(code)

            if desc.startswith(target_name):
                remainder = desc[len(target_name):]
                if (
                    remainder == ""
                    or remainder.startswith("/")
                    or remainder.startswith("-")
                    or remainder.startswith("_")
                    or remainder.isspace()
                ):
                    return int(code)

        return None

    def load_data(self):
        total_loaded = 0

        for sub in self.subjects:
            sub_folder = f"S{sub:03d}"
            sub_path = os.path.join(self.data_dir, sub_folder)

            if not os.path.isdir(sub_path):
                continue

            for run in self.runs:
                edf_file = os.path.join(
                    sub_path,
                    f"{sub_folder}R{run:02d}.edf"
                )

                if not os.path.isfile(edf_file):
                    continue

                try:
                    raw = mne.io.read_raw_edf(
                        edf_file,
                        preload=True,
                        verbose=False
                    )

                    if len(raw.ch_names) < N_CHANNELS:
                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"only {len(raw.ch_names)} channels found; "
                            f"expected {N_CHANNELS}."
                        )
                        continue

                    # Keep first 22 channels, matching the supplied pipeline.
                    raw.pick(raw.ch_names[:N_CHANNELS])
                    raw.resample(FS, npad="auto")

                    events, event_id_dict = mne.events_from_annotations(
                        raw, verbose=False
                    )

                    t1_code = self._find_event_code(
                        event_id_dict, "T1"
                    )
                    t2_code = self._find_event_code(
                        event_id_dict, "T2"
                    )

                    if t1_code is None or t2_code is None:
                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"T1/T2 not found. Available annotations: "
                            f"{list(event_id_dict.keys())}"
                        )
                        continue

                    # IMPORTANT: these are MNE event codes, not class labels.
                    event_selection = {
                        "T1": t1_code,
                        "T2": t2_code
                    }

                    ep = mne.Epochs(
                        raw,
                        events,
                        event_id=event_selection,
                        tmin=self.tmin,
                        tmax=self.tmax - 1.0 / FS,
                        baseline=None,
                        preload=True,
                        reject_by_annotation=True,
                        verbose=False
                    )

                    if len(ep) == 0:
                        print(
                            f"[WARN] {sub_folder} R{run:02d}: "
                            f"T1/T2 found but zero epochs created."
                        )
                        continue

                    data = ep.get_data()
                    actual_codes = ep.events[:, -1]

                    for i in range(len(data)):
                        actual_code = int(actual_codes[i])

                        # Binary ML mapping:
                        # T1 = 0 = Left Hand MI
                        # T2 = 1 = Right Hand MI
                        if actual_code == t1_code:
                            class_label = 0
                        elif actual_code == t2_code:
                            class_label = 1
                        else:
                            continue

                        self.epochs.append(
                            data[i].astype(np.float32)
                        )
                        self.labels.append(int(class_label))
                        self.subject_ids.append(int(sub - 1))
                        self.run_ids.append(int(run))
                        total_loaded += 1

                    print(
                        f"[OK] {sub_folder} R{run:02d} | "
                        f"T1={t1_code}->Left | "
                        f"T2={t2_code}->Right | "
                        f"epochs={len(ep)}"
                    )

                except Exception as e:
                    print(
                        f"[WARN] {sub_folder} R{run:02d}: "
                        f"{type(e).__name__}: {e}"
                    )

        print(
            f"\nDataset loading complete: "
            f"{total_loaded} binary trials"
        )

    def __len__(self):
        return len(self.epochs)

    def __getitem__(self, idx):
        x = torch.tensor(self.epochs[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        s = torch.tensor(self.subject_ids[idx], dtype=torch.long)

        # Per-trial, per-channel Z-score normalization.
        mean = x.mean(dim=1, keepdim=True)
        std = x.std(dim=1, keepdim=True)
        x = (x - mean) / (std + 1e-6)

        return x, y, s


def discover_available_subjects(data_dir, max_subjects=TOTAL_SUBJECTS):
    available = []
    for s in range(1, max_subjects + 1):
        if os.path.isdir(Path(data_dir) / f"S{s:03d}"):
            available.append(s)
    return available


# ------------------------------------------------------------
# Binary dataset sanity check
# ------------------------------------------------------------
available_subjects = discover_available_subjects(
    DATA_DIR, TOTAL_SUBJECTS
)

print("Available subject folders:", len(available_subjects))

if available_subjects:
    test_subject = available_subjects[0]
    print(f"Testing binary loader on S{test_subject:03d}")

    test_dataset = EEGMMIDB_Dataset(
        DATA_DIR,
        subjects=[test_subject]
    )

    print("Number of trials:", len(test_dataset))

    if len(test_dataset) > 0:
        labels = np.array(test_dataset.labels)
        unique, counts = np.unique(labels, return_counts=True)

        print("Class distribution:")
        for c, n in zip(unique, counts):
            print(f"  {c} ({CLASS_NAMES[c]}): {n}")

        x0, y0, s0 = test_dataset[0]
        print("Example tensor shape:", tuple(x0.shape))
        print("Example label:", int(y0))
        print("Example subject ID:", int(s0))
        print("[OK] Binary dataset loader is ready.")


Available subject folders: 109
Testing binary loader on S001
[OK] S001 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S001 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S001 R12 | T1=2->Left | T2=3->Right | epochs=15

Dataset loading complete: 45 binary trials
Number of trials: 45
Class distribution:
  0 (Left Hand MI): 23
  1 (Right Hand MI): 22
Example tensor shape: (22, 1000)
Example label: 1
Example subject ID: 0
[OK] Binary dataset loader is ready.


## CELL 04 — Gradient Reversal and learnable Sinc filter bank

This part starts the feature extractor.

### Gradient Reversal Layer
During forward propagation the GRL behaves like an identity operation. During backpropagation it multiplies the gradient by `-lambda`. Therefore:
- the domain classifier learns to predict the subject;
- the feature extractor receives the **opposite** gradient and is encouraged to make subjects harder to distinguish.

### Learnable Sinc filter bank
The model learns two frequency cutoffs for each of 10 filters. Each filter is constructed as a band-pass Sinc kernel and applied independently to every EEG channel.

Input shape:
`(B, 22, 1000)`

Output shape:
`(B, 10, 22, 1000)`

So the raw temporal signal becomes a learned set of 10 spectral representations. fileciteturn1file0L254-L309

In [9]:
# ============================================================
# CELL 04 — GRL and SincFilterBank
# ============================================================

# -----------------------------
# 2. ARCHITECTURE
# -----------------------------

class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = lambda_grl
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_grl, None

def grl(x, lambda_grl=1.0):
    return GradientReversalLayer.apply(x, lambda_grl)





## CELL 05 — Dynamic graph neural network

`DGNN` builds an adaptive channel graph from the spectral features.

For each trial it:
1. averages over time to obtain channel descriptors;
2. projects the descriptors into query and key spaces;
3. computes pairwise channel similarity;
4. applies softmax to obtain a dynamic adjacency matrix;
5. adds self-connections;
6. degree-normalizes the adjacency matrix;
7. mixes information across EEG channels;
8. projects the resulting 22-node representation into a 64-dimensional spatial feature space.

The returned tensor has shape:
`(B, 10 bands, 64 spatial features, T)`. fileciteturn1file0L312-L343

In [13]:
# ============================================================
# CELL 05 — DYNAMIC GRAPH NEURAL NETWORK (DGNN)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class DGNN(nn.Module):
    """
    Dynamic Graph Neural Network for multi-band EEG.

    Input:
        [B, F, N, T]

        B = batch
        F = number of spectral filters
        N = EEG channels
        T = time samples

    Output:
        [B, F, out_nodes, T]
    """

    def __init__(
        self,
        num_filters=10,
        in_nodes=22,
        out_nodes=64
    ):
        super().__init__()

        self.num_filters = num_filters
        self.in_nodes = in_nodes
        self.out_nodes = out_nodes

        # ----------------------------------------------------
        # Learnable channel-to-spatial projection
        # ----------------------------------------------------

        self.node_projection = nn.Parameter(
            torch.randn(
                num_filters,
                in_nodes,
                out_nodes
            ) * 0.02
        )

        # ----------------------------------------------------
        # Channel feature extraction
        # ----------------------------------------------------

        self.channel_conv = nn.Sequential(

            nn.Conv2d(
                num_filters,
                num_filters,
                kernel_size=(3, 1),
                padding=(1, 0),
                groups=num_filters,
                bias=False
            ),

            nn.BatchNorm2d(
                num_filters
            ),

            nn.ELU(
                inplace=True
            )
        )

        # ----------------------------------------------------
        # Spatial mixing
        # ----------------------------------------------------

        self.spatial_mixing = nn.Conv2d(
            num_filters,
            num_filters,
            kernel_size=1,
            bias=False
        )

        self.bn = nn.BatchNorm2d(
            num_filters
        )

        self.activation = nn.ELU(
            inplace=True
        )

    def forward(self, x):

        # ----------------------------------------------------
        # Input
        #
        # [B, F, N, T]
        # ----------------------------------------------------

        if x.ndim != 4:

            raise ValueError(
                "DGNN expected input "
                "[B, F, N, T], got "
                f"{tuple(x.shape)}"
            )

        B, F_band, N, T = x.shape

        if F_band != self.num_filters:

            raise ValueError(
                f"Expected {self.num_filters} filters, "
                f"got {F_band}"
            )

        if N != self.in_nodes:

            raise ValueError(
                f"Expected {self.in_nodes} EEG channels, "
                f"got {N}"
            )

        # ----------------------------------------------------
        # Local temporal/channel processing
        # ----------------------------------------------------

        h = self.channel_conv(x)

        # ----------------------------------------------------
        # Spatial graph projection
        #
        # h:
        #     [B, F, N, T]
        #
        # projection:
        #     [F, N, D]
        #
        # result:
        #     [B, F, D, T]
        # ----------------------------------------------------

        h = torch.einsum(
            "bfnt,fnd->bfdt",
            h,
            self.node_projection
        )

        # ----------------------------------------------------
        # Spatial feature mixing
        # ----------------------------------------------------

        h = self.spatial_mixing(
            h
        )

        h = self.bn(
            h
        )

        h = self.activation(
            h
        )

        return h


# ============================================================
# CELL 05 CHECK
# ============================================================

print("=" * 70)
print("CELL 05 — DGNN CHECK")
print("=" * 70)

try:

    test_dgnn = DGNN(
        num_filters=NUM_FILTERS,
        in_nodes=N_CHANNELS,
        out_nodes=SPATIAL_DIM
    ).to(DEVICE)

    test_input = torch.randn(
        2,
        NUM_FILTERS,
        N_CHANNELS,
        int(FS * TMAX)
    ).to(DEVICE)

    with torch.no_grad():

        test_output = test_dgnn(
            test_input
        )

    print(
        "Input shape :",
        tuple(test_input.shape)
    )

    print(
        "Output shape:",
        tuple(test_output.shape)
    )

    expected = (
        2,
        NUM_FILTERS,
        SPATIAL_DIM,
        int(FS * TMAX)
    )

    print(
        "Expected    :",
        expected
    )

    assert (
        tuple(test_output.shape)
        ==
        expected
    )

    print()
    print("[OK] DGNN is defined and working.")

except Exception as e:

    print()
    print("[ERROR] DGNN test failed:")
    print(
        type(e).__name__,
        ":",
        e
    )

finally:

    for name in [
        "test_dgnn",
        "test_input",
        "test_output"
    ]:

        if name in globals():
            del globals()[name]

CELL 05 — DGNN CHECK
Input shape : (2, 10, 22, 1000)
Output shape: (2, 10, 64, 1000)
Expected    : (2, 10, 64, 1000)

[OK] DGNN is defined and working.


## CELL 06 — Temporal block and SE attention

The class named `SimplifiedBiMamba` is a bidirectional GRU.

After averaging the 10 spectral bands, the tensor is:
`(B, 64, T)`.

The GRU reads the time axis in both directions and returns another `(B, 64, T)` representation.

`SEAttention` then:
1. averages each feature over time;
2. uses a small bottleneck MLP and sigmoid to generate feature weights;
3. reweights the temporal features;
4. averages over time to produce the final 64-dimensional embedding.

This 64-D representation is the shared feature used by the task, domain, and contrastive objectives. fileciteturn1file0L346-L381

In [14]:
# ============================================================
# CELL 06 — OPTIMIZED TEMPORAL ENCODER + SE ATTENTION
# ============================================================

class TemporalDownsample(nn.Module):
    """
    Lightweight temporal feature extractor.

    Input:
        [B, C, T]

    Output:
        [B, C, T/4] approximately

    This reduces the sequence length before the BiGRU.
    """

    def __init__(self, channels=64):
        super().__init__()

        self.net = nn.Sequential(

            # ------------------------------------------------
            # First temporal convolution
            # ------------------------------------------------
            nn.Conv1d(
                channels,
                channels,
                kernel_size=7,
                stride=2,
                padding=3,
                groups=channels,
                bias=False
            ),

            nn.BatchNorm1d(channels),
            nn.ELU(inplace=True),

            # ------------------------------------------------
            # Second temporal convolution
            # ------------------------------------------------
            nn.Conv1d(
                channels,
                channels,
                kernel_size=7,
                stride=2,
                padding=3,
                groups=channels,
                bias=False
            ),

            nn.BatchNorm1d(channels),
            nn.ELU(inplace=True),

            # ------------------------------------------------
            # Pointwise feature mixing
            # ------------------------------------------------
            nn.Conv1d(
                channels,
                channels,
                kernel_size=1,
                bias=False
            ),

            nn.BatchNorm1d(channels),
            nn.ELU(inplace=True)
        )

    def forward(self, x):

        # x:
        # [B, C, T]

        return self.net(x)


# ============================================================
# BIDIRECTIONAL TEMPORAL GRU
# ============================================================

class SimplifiedBiMamba(nn.Module):
    """
    The original implementation calls this BiMamba,
    but technically it is a Bidirectional GRU.

    Input:
        [B, D, T]

    Output:
        [B, D, T]
    """

    def __init__(self, d_model=64):
        super().__init__()

        self.ssm = nn.GRU(
            input_size=d_model,
            hidden_size=d_model // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):

        # [B, D, T] -> [B, T, D]
        x_seq = x.transpose(1, 2)

        out, _ = self.ssm(x_seq)

        # [B, T, D] -> [B, D, T]
        return out.transpose(1, 2)


# ============================================================
# SE ATTENTION
# ============================================================

class SEAttention(nn.Module):

    def __init__(
        self,
        channel=64,
        reduction=16
    ):
        super().__init__()

        hidden_dim = max(
            1,
            channel // reduction
        )

        self.fc = nn.Sequential(

            nn.Linear(
                channel,
                hidden_dim,
                bias=False
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                hidden_dim,
                channel,
                bias=False
            ),

            nn.Sigmoid()
        )

    def forward(self, x):

        # x = [B, C, T]

        b, c, t = x.size()

        # Global temporal pooling
        y = x.mean(dim=2)

        # Channel attention
        weights = self.fc(y)

        weights = weights.view(
            b,
            c,
            1
        )

        # Re-weight channels
        x = x * weights

        # Final temporal pooling
        return x.mean(dim=2)


# ============================================================
# VERIFY
# ============================================================

print("=" * 70)
print("CELL 06 CHECK")
print("=" * 70)

test_x = torch.randn(
    2,
    SPATIAL_DIM,
    int(FS * TMAX)
).to(DEVICE)

with torch.no_grad():

    temporal_downsample = TemporalDownsample(
        SPATIAL_DIM
    ).to(DEVICE)

    temporal_gru = SimplifiedBiMamba(
        SPATIAL_DIM
    ).to(DEVICE)

    se_attention = SEAttention(
        SPATIAL_DIM
    ).to(DEVICE)

    x_down = temporal_downsample(
        test_x
    )

    x_gru = temporal_gru(
        x_down
    )

    x_se = se_attention(
        x_gru
    )

print("Input               :", tuple(test_x.shape))
print("After downsampling  :", tuple(x_down.shape))
print("After BiGRU         :", tuple(x_gru.shape))
print("After SE attention  :", tuple(x_se.shape))

print()
print("[OK] Optimized temporal module ready.")

CELL 06 CHECK
Input               : (2, 64, 1000)
After downsampling  : (2, 64, 250)
After BiGRU         : (2, 64, 250)
After SE attention  : (2, 64)

[OK] Optimized temporal module ready.


## CELL 07 — Full S³ Binary Domain Adaptation model

`S3MambaDA` connects the complete feature extractor and the three training heads:

**Input → Sinc → DGNN → band pooling → BiGRU → SE attention → 64-D embedding**

Then:
- **Classifier:** 64 → 4 class logits.
- **Domain classifier:** GRL → 64 → 32 → 109 subject logits.
- **SupCon projection:** 64 → 128 → 128, followed by L2 normalization.

The forward pass therefore returns three objects:
`class_logits, domain_logits, z_proj`. fileciteturn1file0L384-L436

In [15]:
# ============================================================
# CELL 07 — S3MambaDA BINARY MODEL
# ============================================================

class S3MambaDA(nn.Module):

    def __init__(
        self,
        num_classes=2,
        num_subjects=109
    ):
        super().__init__()

        # ----------------------------------------------------
        # SPECTRAL
        # ----------------------------------------------------

        self.sinc_filter = SincFilterBank(
            in_channels=N_CHANNELS,
            num_filters=NUM_FILTERS,
            kernel_size=SINC_KERNEL,
            sample_rate=int(FS)
        )

        # ----------------------------------------------------
        # SPATIAL
        # ----------------------------------------------------

        self.dgnn = DGNN(
            num_filters=NUM_FILTERS,
            in_nodes=N_CHANNELS,
            out_nodes=SPATIAL_DIM
        )

        # ----------------------------------------------------
        # TEMPORAL DOWNSAMPLING
        # ----------------------------------------------------

        self.temporal_downsample = TemporalDownsample(
            channels=SPATIAL_DIM
        )

        # ----------------------------------------------------
        # TEMPORAL MODEL
        # ----------------------------------------------------

        self.mamba = SimplifiedBiMamba(
            d_model=SPATIAL_DIM
        )

        # ----------------------------------------------------
        # ATTENTION
        # ----------------------------------------------------

        self.se_attention = SEAttention(
            channel=SPATIAL_DIM
        )

        # ----------------------------------------------------
        # BINARY CLASSIFIER
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.BatchNorm1d(
                SPATIAL_DIM
            ),

            nn.Dropout(
                p=0.20
            ),

            nn.Linear(
                SPATIAL_DIM,
                num_classes
            )
        )

        # ----------------------------------------------------
        # DOMAIN CLASSIFIER
        # ----------------------------------------------------

        self.domain_classifier = nn.Sequential(

            nn.Linear(
                SPATIAL_DIM,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                p=0.20
            ),

            nn.Linear(
                64,
                num_subjects
            )
        )

        # ----------------------------------------------------
        # SUPCON PROJECTION
        # ----------------------------------------------------

        self.supcon_proj = nn.Sequential(

            nn.Linear(
                SPATIAL_DIM,
                128
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                p=0.10
            ),

            nn.Linear(
                128,
                128
            )
        )

    def forward(
        self,
        x,
        lambda_grl=0.0
    ):

        # ====================================================
        # 1. SPECTRAL
        # ====================================================

        x = self.sinc_filter(x)

        # [B, F, N, T]

        # ====================================================
        # 2. SPATIAL GRAPH
        # ====================================================

        x = self.dgnn(x)

        # [B, F, D, T]

        # ====================================================
        # 3. BAND FUSION
        # ====================================================

        x = x.mean(
            dim=1
        )

        # [B, D, T]

        # ====================================================
        # 4. TEMPORAL DOWNSAMPLING
        # ====================================================

        x = self.temporal_downsample(
            x
        )

        # [B, D, T/4]

        # ====================================================
        # 5. BIDIRECTIONAL TEMPORAL MODEL
        # ====================================================

        x = self.mamba(
            x
        )

        # [B, D, T/4]

        # ====================================================
        # 6. SE ATTENTION
        # ====================================================

        z = self.se_attention(
            x
        )

        # [B, D]

        # ====================================================
        # 7. BINARY CLASSIFICATION
        # ====================================================

        class_logits = self.classifier(
            z
        )

        # [B, 2]

        # ====================================================
        # 8. DOMAIN ADVERSARIAL BRANCH
        # ====================================================

        z_domain = grl(
            z,
            lambda_grl
        )

        domain_logits = (
            self.domain_classifier(
                z_domain
            )
        )

        # [B, 109]

        # ====================================================
        # 9. SUPCON
        # ====================================================

        z_proj = self.supcon_proj(
            z
        )

        z_proj = F.normalize(
            z_proj,
            p=2,
            dim=1
        )

        # [B, 128]

        return (
            class_logits,
            domain_logits,
            z_proj
        )


# ============================================================
# CELL 07 CHECK
# ============================================================

print("=" * 70)
print("CELL 07 CHECK")
print("=" * 70)

try:

    model_test = S3MambaDA(
        num_classes=2,
        num_subjects=DOMAIN_CLASSES
    ).to(DEVICE)

    test_input = torch.randn(
        2,
        N_CHANNELS,
        int(FS * TMAX)
    ).to(DEVICE)

    with torch.no_grad():

        (
            class_logits,
            domain_logits,
            z_proj
        ) = model_test(
            test_input,
            lambda_grl=0.0
        )

    print(
        "Input          :",
        tuple(test_input.shape)
    )

    print(
        "Class logits   :",
        tuple(class_logits.shape)
    )

    print(
        "Domain logits  :",
        tuple(domain_logits.shape)
    )

    print(
        "Projection     :",
        tuple(z_proj.shape)
    )

    print(
        "Parameters     :",
        f"{sum(p.numel() for p in model_test.parameters()):,}"
    )

    # --------------------------------------------------------
    # Shape assertions
    # --------------------------------------------------------

    assert class_logits.shape == (
        2,
        2
    )

    assert domain_logits.shape == (
        2,
        DOMAIN_CLASSES
    )

    assert z_proj.shape == (
        2,
        128
    )

    print()
    print(
        "[OK] S3MambaDA binary model passed smoke test."
    )

except Exception as e:

    print()
    print(
        "[ERROR] Cell 07 failed:"
    )

    print(
        type(e).__name__,
        ":",
        e
    )

finally:

    for name in [
        "model_test",
        "test_input",
        "class_logits",
        "domain_logits",
        "z_proj"
    ]:

        if name in globals():
            del globals()[name]

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

CELL 07 CHECK
Input          : (2, 22, 1000)
Class logits   : (2, 2)
Domain logits  : (2, 109)
Projection     : (2, 128)
Parameters     : 75,309

[OK] S3MambaDA binary model passed smoke test.


NameError: name 'gc' is not defined

## CELL 08 — Losses and AdaBN adaptation

The training objective is the sum of three components:

`Total = Classification Loss + 1.0 × Domain Loss + 0.5 × SupCon Loss`

The classification loss uses cross-entropy with label smoothing. The domain loss predicts the subject identity. The supervised contrastive loss uses class labels so trials from the same class act as positives.

After training on source subjects, `apply_adabn()` collects unlabeled target trials and refreshes BatchNorm running statistics from target data. This is a **test-time statistics adaptation** step rather than supervised target-label fine-tuning. fileciteturn1file0L442-L512

In [41]:
# ============================================================
# CELL 08 — SupCon loss and AdaBN adaptation
# ============================================================

# -----------------------------
# 3. LOSSES
# -----------------------------
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]

        sim = torch.matmul(features, features.T) / self.temperature

        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        logits_mask = torch.ones_like(mask)
        logits_mask.fill_diagonal_(0)
        mask = mask * logits_mask

        exp_logits = torch.exp(sim) * logits_mask
        log_prob = sim - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

        positives = mask.sum(1)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (positives + 1e-6)

        valid = positives > 0
        if valid.any():
            return -mean_log_prob_pos[valid].mean()
        return torch.zeros((), device=device, requires_grad=True)


def apply_adabn(model, target_dataloader, device, adaptation_trials=20):
    model.eval()

    target_samples = []
    trials_count = 0

    for x, _, _ in target_dataloader:
        target_samples.append(x)
        trials_count += x.size(0)
        if trials_count >= adaptation_trials:
            break

    if not target_samples:
        return model

    target_x = torch.cat(target_samples, dim=0)[:adaptation_trials].to(device)

    bn_modules = [
        m for m in model.modules()
        if isinstance(m, nn.modules.batchnorm._BatchNorm)
    ]

    if not bn_modules:
        return model

    saved = []
    for module in bn_modules:
        saved.append((module.training, module.momentum))
        module.reset_running_stats()
        module.momentum = 1.0
        module.train()

    with torch.no_grad():
        _ = model(target_x, lambda_grl=0.0)

    for module, (was_training, old_momentum) in zip(bn_modules, saved):
        module.momentum = 0.1
        module.eval()

    model.eval()
    return model


## CELL 09 — Model smoke test

Before launching the expensive experiment, this cell creates a random tensor with the expected EEG size and verifies that the complete model executes.

Expected:
- input `(2, 22, 1000)`
- 4 class logits
- 109 subject/domain logits
- 128-dimensional contrastive projection

It also reports the parameter count. fileciteturn1file0L515-L532

In [42]:
# ============================================================
# CELL 09 — Model smoke test
# ============================================================

# Run this cell only AFTER Cells 01–08 have been executed.
# This guard turns a confusing NameError into a direct instruction.
required = [
    "torch", "nn", "F", "N_CLASSES", "DOMAIN_CLASSES", "DEVICE",
    "N_CHANNELS", "FS", "TMAX", "S3MambaDA"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing notebook definitions: " + ", ".join(missing) +
        ". Run Cells 01 through 08 in order before running Cell 09."
    )


def smoke_test():
    model = S3MambaDA(
        num_classes=N_CLASSES,
        num_subjects=DOMAIN_CLASSES
    ).to(DEVICE)

    x = torch.randn(
        2,
        N_CHANNELS,
        int(FS * TMAX),
        device=DEVICE
    )

    with torch.no_grad():
        class_logits, domain_logits, z_proj = model(
            x,
            lambda_grl=0.0
        )

    print("Smoke test PASSED")
    print("  input:          ", tuple(x.shape))
    print("  class logits:   ", tuple(class_logits.shape))
    print("  domain logits:  ", tuple(domain_logits.shape))
    print("  projection:     ", tuple(z_proj.shape))
    print("  parameters:     ", sum(p.numel() for p in model.parameters()))


smoke_test()


Smoke test PASSED
  input:           (2, 22, 1000)
  class logits:    (2, 2)
  domain logits:   (2, 109)
  projection:      (2, 128)
  parameters:      62751


## CELL 10 — Training and subject-independent evaluation

`evaluate_large_scale_loso()` is the main experiment.

For each selected held-out subject:
1. choose the test subject;
2. sample up to 99 other available subjects for training;
3. load all selected runs for those subjects;
4. train a fresh model for 100 epochs;
5. ramp the GRL strength from approximately 0 toward 1 using a logistic schedule;
6. optimize classification + domain + contrastive losses;
7. apply AdaBN using all available target trials;
8. evaluate the target subject without labels during adaptation;
9. collect predictions, probabilities, embeddings, and per-epoch losses.

**Important terminology:** the function is named `evaluate_large_scale_loso`, but with 109 subjects it samples 10 test subjects, and each fold uses 99 of the other 108 subjects for training. The remaining 9 subjects are not used in that fold. Therefore this implementation is better described as a **10-fold sampled subject-held-out evaluation** rather than complete 109-fold LOSO. fileciteturn1file0L538-L581

In [43]:
# ============================================================
# CELL 10 — OPTIMIZED BINARY MI CONFIGURATION
# ============================================================

import os
import gc
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


# ------------------------------------------------------------
# EXACT TARGET SUBJECTS
# ------------------------------------------------------------

TEST_SUBJECTS = [
    4,
    15,
    23,
    29,
    31,
    42,
    55,
    71,
    82,
    95
]


# ------------------------------------------------------------
# Binary task
# ------------------------------------------------------------

N_CLASSES = 2

CLASS_NAMES = [
    "Left",
    "Right"
]


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

BATCH_SIZE = 64

NUM_EPOCHS = 120

LR = 3e-4

MIN_LR = 1e-6

WEIGHT_DECAY = 1e-4

GRAD_CLIP = 1.0


# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

LABEL_SMOOTHING = 0.02

SUPCON_WEIGHT = 0.05

SUPCON_TEMP = 0.10


# ------------------------------------------------------------
# Domain adaptation
#
# Important:
# We start with ZERO domain pressure and gradually increase it.
# ------------------------------------------------------------

DOMAIN_WEIGHT_MAX = 0.08

DOMAIN_WARMUP = 25


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

NUM_FILTERS = 10

SPATIAL_DIM = 64


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

VAL_FRACTION = 0.15

VAL_SEED = 123


# ------------------------------------------------------------
# Early stopping
# ------------------------------------------------------------

PATIENCE = 20


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

RESULTS_DIR = Path(
    "./results_binary_optimized_v2"
)

FIG_DIR = (
    RESULTS_DIR / "figures"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 80)
print("OPTIMIZED BINARY LEFT-vs-RIGHT MI")
print("=" * 80)

print(
    "Target subjects:",
    [
        f"S{s:03d}"
        for s in TEST_SUBJECTS
    ]
)

print()
print("Epochs          :", NUM_EPOCHS)
print("Batch size      :", BATCH_SIZE)
print("Learning rate   :", LR)
print("Weight decay    :", WEIGHT_DECAY)
print("Domain max      :", DOMAIN_WEIGHT_MAX)
print("Domain warmup   :", DOMAIN_WARMUP)
print("SupCon weight   :", SUPCON_WEIGHT)
print("Validation frac :", VAL_FRACTION)
print()

## CELL 11 — Architecture and training-loss figures

These functions create paper-style figures from the experiment outputs.

`plot_architecture()` draws the conceptual network.
`plot_training_curves()` aggregates losses across folds by epoch and plots total, classification, domain, and SupCon losses. fileciteturn1file0L808-L894

In [44]:
# ============================================================
# CELL 11 — SUBJECT-LEVEL SOURCE VALIDATION SPLIT
# ============================================================

def make_subject_validation_split(
    subjects,
    val_fraction=0.15,
    seed=123
):

    subjects = list(
        sorted(subjects)
    )

    rng = np.random.RandomState(
        seed
    )

    rng.shuffle(subjects)

    n_val = max(
        1,
        int(
            round(
                len(subjects)
                *
                val_fraction
            )
        )
    )

    val_subjects = sorted(
        subjects[:n_val]
    )

    train_subjects = sorted(
        subjects[n_val:]
    )

    return (
        train_subjects,
        val_subjects
    )


print("[OK] Validation split function ready.")

## CELL 12 — Performance figures

These functions visualize:
- held-out-subject accuracy;
- normalized confusion matrix;
- precision, recall, and F1 for the four classes;
- one-vs-rest ROC curves and AUC.

They read the CSV artifacts generated by the evaluation function. fileciteturn1file0L897-L1007

In [45]:
# ============================================================
# CELL 12 — CLASS-BALANCED SAMPLER
# ============================================================

def make_balanced_sampler(dataset):

    labels = np.asarray(
        dataset.labels,
        dtype=np.int64
    )

    classes, counts = np.unique(
        labels,
        return_counts=True
    )

    class_weights = {
        int(c): 1.0 / float(n)
        for c, n in zip(
            classes,
            counts
        )
    }

    sample_weights = np.asarray(
        [
            class_weights[int(y)]
            for y in labels
        ],
        dtype=np.float64
    )

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(
            sample_weights,
            dtype=torch.double
        ),
        num_samples=len(
            sample_weights
        ),
        replacement=True
    )

    return sampler


print("[OK] Balanced sampler ready.")

## CELL 13 — Embedding visualization and figure generation

`plot_tsne()` reduces the saved 128-D contrastive embeddings to 2-D using t-SNE and colors points by the true EEG class.

`generate_all_figures()` first generates the architecture figure, then checks whether all result CSV files exist. If they do, it generates the remaining six figures and saves them under `results_s3_da/figures/`. fileciteturn2file0L10-L75

In [46]:
# ============================================================
# CELL 13 — EEG AUGMENTATION
# ============================================================

def augment_eeg(
    x,
    noise_std=0.015,
    channel_dropout=0.05,
    amplitude_jitter=0.05
):

    if not torch.is_tensor(x):
        return x

    x = x.clone()

    # --------------------------------------------------------
    # Small amplitude scaling
    # --------------------------------------------------------

    if amplitude_jitter > 0:

        scale = (
            1.0
            +
            torch.randn(
                x.size(0),
                1,
                1,
                device=x.device
            )
            *
            amplitude_jitter
        )

        x = x * scale


    # --------------------------------------------------------
    # Small Gaussian noise
    # --------------------------------------------------------

    if noise_std > 0:

        noise = (
            torch.randn_like(x)
            *
            noise_std
        )

        x = x + noise


    # --------------------------------------------------------
    # Very small channel dropout
    # --------------------------------------------------------

    if channel_dropout > 0:

        mask = (
            torch.rand(
                x.size(0),
                x.size(1),
                1,
                device=x.device
            )
            >
            channel_dropout
        )

        x = x * mask


    return x


print("[OK] EEG augmentation ready.")

## CELL 14 — Run the experiment

Run this cell only after the smoke test succeeds.

For a cheap pipeline check, temporarily set `NUM_EPOCHS` to a small value such as 1–2 in CELL 02. For the full source configuration, restore it to 100.

The original notebook calls figure generation immediately, but the actual training call is left commented out. This cell makes the intended execution order explicit.

In [ ]:
# ============================================================
# CELL 14 — GRADUAL DOMAIN-ADVERSARIAL SCHEDULE
# ============================================================

def get_domain_weight(
    epoch,
    max_weight=DOMAIN_WEIGHT_MAX,
    warmup_epochs=DOMAIN_WARMUP
):

    if epoch < warmup_epochs:
        return 0.0

    p = (
        epoch - warmup_epochs
    ) / max(
        1,
        NUM_EPOCHS - warmup_epochs
    )

    p = np.clip(
        p,
        0.0,
        1.0
    )

    # Smooth ramp
    weight = (
        max_weight
        *
        (
            0.5
            -
            0.5
            *
            np.cos(
                np.pi * p
            )
        )
    )

    return float(weight)


def get_grl_lambda(
    epoch,
    total_epochs
):

    # No GRL initially.
    if epoch < 10:
        return 0.0

    p = (
        epoch - 10
    ) / max(
        1,
        total_epochs - 10
    )

    p = np.clip(
        p,
        0.0,
        1.0
    )

    return float(
        2.0
        /
        (
            1.0
            +
            np.exp(
                -10.0 * p
            )
        )
        -
        1.0
    )


for e in [0, 5, 10, 20, 40, 80, 119]:

    print(
        f"Epoch {e+1:03d} | "
        f"DomainWeight={get_domain_weight(e):.4f} | "
        f"GRL={get_grl_lambda(e, NUM_EPOCHS):.4f}"
    )

Starting binary subject-independent experiment...
Task: Left Hand MI vs Right Hand MI
Chance level: 50.00%
Available subject folders: 109
FOLD 1/10 | TEST SUBJECT S082
[OK] S096 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S096 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S096 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S070 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S070 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S070 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S012 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S012 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S012 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S076 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S076 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S076 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S055 R04 | T1=2->Left | T2=3->Right | epochs=15
[OK] S055 R08 | T1=2->Left | T2=3->Right | epochs=15
[OK] S055 R12 | T1=2->Left | T2=3->Right | epochs=15
[OK] S005 R04 | T1=2->Left | T2=3->Ri

## CELL 15 — Export paper-ready results text

This helper reads `fold_metrics.csv` and produces a human-readable `paper_results.txt` containing:
- number of completed folds;
- held-out subject IDs;
- mean ± standard deviation of accuracy;
- mean ± standard deviation of Cohen's kappa;
- the full per-fold metrics table.

In [34]:
# ============================================================
# CELL 15 — VALIDATION EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_binary_model(
    model,
    loader,
    device
):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    for batch in loader:

        x = batch[0]
        y = batch[1]

        x = x.to(device)
        y = y.to(device)

        outputs = model(
            x,
            lambda_grl=0.0
        )

        logits = outputs[0]

        probs = torch.softmax(
            logits,
            dim=1
        )

        pred = torch.argmax(
            probs,
            dim=1
        )

        y_true.extend(
            y.cpu().numpy().tolist()
        )

        y_pred.extend(
            pred.cpu().numpy().tolist()
        )

        y_prob.extend(
            probs[:, 1]
            .cpu()
            .numpy()
            .tolist()
        )

    acc = accuracy_score(
        y_true,
        y_pred
    )

    bal_acc = balanced_accuracy_score(
        y_true,
        y_pred
    )

    kappa = cohen_kappa_score(
        y_true,
        y_pred
    )

    try:

        auc = roc_auc_score(
            y_true,
            y_prob
        )

    except Exception:

        auc = np.nan

    return {
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "kappa": kappa,
        "auc": auc,
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "y_prob": np.asarray(y_prob)
    }


print("[OK] Evaluation function ready.")

BINARY LEFT-vs-RIGHT MOTOR-IMAGERY RESULTS
Completed folds: 10
Test subjects: S082, S015, S004, S095, S036, S032, S029, S018, S014, S087

Pooled Accuracy: 70.67%
Mean Fold Accuracy: 70.67% ± 10.41%
Mean Balanced Accuracy: 70.70%
Mean Cohen's Kappa: 0.4136
Mean ROC-AUC: 0.7448
Binary chance level: 50.00%

Per-fold results:
 fold  test_subject  n_train_trials  n_test_trials  accuracy  balanced_accuracy    kappa  roc_auc  precision_macro  recall_macro  f1_macro
    1            82            4467             45  0.800000           0.800395 0.600197 0.729249         0.800395      0.800395  0.800000
    2            15            4467             45  0.777778           0.777668 0.555336 0.867589         0.777668      0.777668  0.777668
    3             4            4467             45  0.733333           0.733202 0.466403 0.822134         0.733202      0.733202  0.733202
    4            95            4467             45  0.600000           0.600791 0.201183 0.636364         0.601190      

In [ ]:
# ============================================================
# CELL 16 — OPTIMIZED TRAINING FUNCTION
# ============================================================

def train_optimized_model(
    model,
    train_loader,
    val_loader,
    train_dataset,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY
):

    model = model.to(DEVICE)

    # --------------------------------------------------------
    # Losses
    # --------------------------------------------------------

    criterion_cls = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING
    )

    criterion_domain = nn.CrossEntropyLoss()

    criterion_supcon = SupConLoss(
        temperature=SUPCON_TEMP
    )

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # --------------------------------------------------------
    # Warmup + cosine decay
    # --------------------------------------------------------

    warmup_epochs = 8

    def lr_lambda(epoch):

        if epoch < warmup_epochs:

            return (
                0.15
                +
                0.85
                *
                (
                    epoch + 1
                )
                /
                warmup_epochs
            )

        progress = (
            epoch - warmup_epochs
        ) / max(
            1,
            num_epochs - warmup_epochs
        )

        return (
            MIN_LR / lr
            +
            0.5
            *
            (
                1
                -
                MIN_LR / lr
            )
            *
            (
                1
                +
                np.cos(
                    np.pi * progress
                )
            )
        )

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda
    )

    # --------------------------------------------------------
    # AMP
    # --------------------------------------------------------

    use_amp = (
        DEVICE.type == "cuda"
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )

    # --------------------------------------------------------
    # Best checkpoint
    # --------------------------------------------------------

    best_score = -np.inf

    best_state = None

    patience_counter = 0

    history = []

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    for epoch in range(num_epochs):

        model.train()

        running_loss = 0.0
        running_cls = 0.0
        running_domain = 0.0
        running_supcon = 0.0

        total_seen = 0

        domain_weight = get_domain_weight(
            epoch
        )

        grl_lambda = get_grl_lambda(
            epoch,
            num_epochs
        )

        for batch in train_loader:

            x = batch[0]
            y = batch[1]
            subject = batch[2]

            x = x.to(DEVICE)
            y = y.to(DEVICE)
            subject = subject.to(DEVICE)

            # ------------------------------------------------
            # Augmentation
            # ------------------------------------------------

            if np.random.rand() < 0.65:

                x_input = augment_eeg(
                    x
                )

            else:

                x_input = x

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=use_amp
            ):

                outputs = model(
                    x_input,
                    lambda_grl=grl_lambda
                )

                logits = outputs[0]
                domain_logits = outputs[1]
                z_proj = outputs[2]

                # --------------------------------------------
                # Main classification objective
                # --------------------------------------------

                loss_cls = criterion_cls(
                    logits,
                    y
                )

                # --------------------------------------------
                # Domain objective
                # --------------------------------------------

                loss_domain = criterion_domain(
                    domain_logits,
                    subject
                )

                # --------------------------------------------
                # Supervised contrastive objective
                # --------------------------------------------

                loss_supcon = criterion_supcon(
                    z_proj,
                    y
                )

                # --------------------------------------------
                # Total
                # --------------------------------------------

                loss = (
                    loss_cls
                    +
                    domain_weight
                    *
                    loss_domain
                    +
                    SUPCON_WEIGHT
                    *
                    loss_supcon
                )

            # ------------------------------------------------
            # Backpropagation
            # ------------------------------------------------

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            bs = x.size(0)

            total_seen += bs

            running_loss += (
                loss.item()
                *
                bs
            )

            running_cls += (
                loss_cls.item()
                *
                bs
            )

            running_domain += (
                loss_domain.item()
                *
                bs
            )

            running_supcon += (
                loss_supcon.item()
                *
                bs
            )

        scheduler.step()

        # ----------------------------------------------------
        # Averages
        # ----------------------------------------------------

        avg_loss = (
            running_loss
            /
            max(
                1,
                total_seen
            )
        )

        avg_cls = (
            running_cls
            /
            max(
                1,
                total_seen
            )
        )

        avg_domain = (
            running_domain
            /
            max(
                1,
                total_seen
            )
        )

        avg_supcon = (
            running_supcon
            /
            max(
                1,
                total_seen
            )
        )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        val_metrics = evaluate_binary_model(
            model,
            val_loader,
            DEVICE
        )

        val_score = (
            0.60
            *
            val_metrics["balanced_accuracy"]
            +
            0.25
            *
            val_metrics["auc"]
            +
            0.15
            *
            val_metrics["kappa"]
        )

        history.append({

            "epoch":
                epoch + 1,

            "train_loss":
                avg_loss,

            "classification_loss":
                avg_cls,

            "domain_loss":
                avg_domain,

            "supcon_loss":
                avg_supcon,

            "domain_weight":
                domain_weight,

            "grl_lambda":
                grl_lambda,

            "lr":
                optimizer.param_groups[0]["lr"],

            "val_accuracy":
                val_metrics["accuracy"],

            "val_balanced_accuracy":
                val_metrics["balanced_accuracy"],

            "val_kappa":
                val_metrics["kappa"],

            "val_auc":
                val_metrics["auc"],

            "val_score":
                val_score
        })

        # ----------------------------------------------------
        # Checkpoint
        # ----------------------------------------------------

        if val_score > best_score:

            best_score = val_score

            best_state = {
                k:
                v.detach()
                .cpu()
                .clone()

                for k, v
                in model.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------

        if (
            epoch == 0
            or
            (epoch + 1) % 10 == 0
        ):

            print(
                f"Epoch {epoch+1:03d}/{num_epochs} | "
                f"Loss={avg_loss:.4f} | "
                f"Cls={avg_cls:.4f} | "
                f"Dom={domain_weight:.3f} | "
                f"ValAcc={val_metrics['accuracy']*100:.2f}% | "
                f"ValBal={val_metrics['balanced_accuracy']*100:.2f}% | "
                f"ValAUC={val_metrics['auc']:.4f}"
            )

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if patience_counter >= PATIENCE:

            print(
                f"[EARLY STOP] "
                f"epoch {epoch+1}"
            )

            break

    # --------------------------------------------------------
    # Restore best checkpoint
    # --------------------------------------------------------

    if best_state is not None:

        model.load_state_dict(
            best_state
        )

    history_df = pd.DataFrame(
        history
    )

    return (
        model,
        history_df,
        best_score
    )


print("[OK] Optimized training function ready.")

In [ ]:
# ============================================================
# CELL 17 — EXACT 10-SUBJECT OPTIMIZED EXPERIMENT
# ============================================================

def run_optimized_binary_experiment():

    seed_everything(SEED)

    available_subjects = discover_available_subjects(
        DATA_DIR,
        TOTAL_SUBJECTS
    )

    print()
    print(
        "Available subject folders:",
        len(available_subjects)
    )

    # --------------------------------------------------------
    # Check requested subjects
    # --------------------------------------------------------

    missing = [
        s
        for s in TEST_SUBJECTS
        if s not in available_subjects
    ]

    if missing:

        raise RuntimeError(
            "Missing requested subjects: "
            +
            ", ".join(
                f"S{s:03d}"
                for s in missing
            )
        )

    fold_results = []
    prediction_rows = []
    history_rows = []

    # --------------------------------------------------------
    # Ten target subjects
    # --------------------------------------------------------

    for fold, test_subject in enumerate(
        TEST_SUBJECTS,
        start=1
    ):

        print()
        print("=" * 80)
        print(
            f"FOLD {fold}/{len(TEST_SUBJECTS)} "
            f"| TEST SUBJECT S{test_subject:03d}"
        )
        print("=" * 80)

        # ----------------------------------------------------
        # Source subjects
        # ----------------------------------------------------

        source_subjects = [
            s
            for s in available_subjects
            if s != test_subject
        ]

        # ----------------------------------------------------
        # Internal subject-level validation
        # ----------------------------------------------------

        train_subjects, val_subjects = (
            make_subject_validation_split(
                source_subjects,
                VAL_FRACTION,
                VAL_SEED + fold
            )
        )

        print(
            f"Train subjects: {len(train_subjects)}"
        )

        print(
            f"Validation subjects: {len(val_subjects)}"
        )

        print(
            "Validation:",
            [
                f"S{s:03d}"
                for s in val_subjects
            ]
        )

        # ----------------------------------------------------
        # Load datasets
        # ----------------------------------------------------

        train_dataset = EEGMMIDB_Dataset(
            DATA_DIR,
            subjects=train_subjects
        )

        val_dataset = EEGMMIDB_Dataset(
            DATA_DIR,
            subjects=val_subjects
        )

        test_dataset = EEGMMIDB_Dataset(
            DATA_DIR,
            subjects=[test_subject]
        )

        print()
        print(
            "Train trials:",
            len(train_dataset)
        )

        print(
            "Validation trials:",
            len(val_dataset)
        )

        print(
            "Test trials:",
            len(test_dataset)
        )

        # ----------------------------------------------------
        # Verify target
        # ----------------------------------------------------

        test_labels = np.asarray(
            test_dataset.labels
        )

        unique_test = np.unique(
            test_labels
        )

        print(
            "Target classes:",
            unique_test.tolist()
        )

        if len(unique_test) != 2:

            raise RuntimeError(
                f"S{test_subject:03d} "
                "does not contain both "
                "Left and Right classes."
            )

        # ----------------------------------------------------
        # Balanced sampler
        # ----------------------------------------------------

        sampler = make_balanced_sampler(
            train_dataset
        )

        # ----------------------------------------------------
        # DataLoaders
        # ----------------------------------------------------

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=0,
            drop_last=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        seed_everything(
            SEED + fold
        )

        model = S3MambaDA(
            num_classes=2,
            num_subjects=DOMAIN_CLASSES
        ).to(DEVICE)

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model, history_df, best_score = (
            train_optimized_model(
                model,
                train_loader,
                val_loader,
                train_dataset
            )
        )

        history_df["fold"] = fold
        history_df["test_subject"] = test_subject

        history_rows.append(
            history_df
        )

        # ----------------------------------------------------
        # Final target evaluation
        # ----------------------------------------------------

        metrics = evaluate_binary_model(
            model,
            test_loader,
            DEVICE
        )

        # ----------------------------------------------------
        # Store fold metrics
        # --------------------------------------------------------

        row = {

            "fold":
                fold,

            "test_subject":
                test_subject,

            "n_train_subjects":
                len(train_subjects),

            "n_validation_subjects":
                len(val_subjects),

            "n_train_trials":
                len(train_dataset),

            "n_val_trials":
                len(val_dataset),

            "n_test_trials":
                len(test_dataset),

            "accuracy":
                metrics["accuracy"],

            "balanced_accuracy":
                metrics["balanced_accuracy"],

            "kappa":
                metrics["kappa"],

            "roc_auc":
                metrics["auc"],

            "precision_macro":
                precision_score(
                    metrics["y_true"],
                    metrics["y_pred"],
                    average="macro",
                    zero_division=0
                ),

            "recall_macro":
                recall_score(
                    metrics["y_true"],
                    metrics["y_pred"],
                    average="macro",
                    zero_division=0
                ),

            "f1_macro":
                f1_score(
                    metrics["y_true"],
                    metrics["y_pred"],
                    average="macro",
                    zero_division=0
                )
        }

        fold_results.append(
            row
        )

        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        for i in range(
            len(metrics["y_true"])
        ):

            prediction_rows.append({

                "fold":
                    fold,

                "test_subject":
                    test_subject,

                "true_label":
                    int(
                        metrics["y_true"][i]
                    ),

                "pred_label":
                    int(
                        metrics["y_pred"][i]
                    ),

                "prob_right":
                    float(
                        metrics["y_prob"][i]
                    ),

                "prob_left":
                    float(
                        1.0
                        -
                        metrics["y_prob"][i]
                    )
            })

        # ----------------------------------------------------
        # Print result
        # ----------------------------------------------------

        print()
        print(
            f"S{test_subject:03d} RESULTS"
        )

        print(
            f"Accuracy       : "
            f"{metrics['accuracy']*100:.2f}%"
        )

        print(
            f"Balanced Acc   : "
            f"{metrics['balanced_accuracy']*100:.2f}%"
        )

        print(
            f"Kappa          : "
            f"{metrics['kappa']:.4f}"
        )

        print(
            f"ROC-AUC        : "
            f"{metrics['auc']:.4f}"
        )

        print()

        print(
            confusion_matrix(
                metrics["y_true"],
                metrics["y_pred"],
                labels=[0, 1]
            )
        )

        # ----------------------------------------------------
        # Cleanup
        # ----------------------------------------------------

        del model
        del train_loader
        del val_loader
        del test_loader
        del train_dataset
        del val_dataset
        del test_dataset

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ========================================================
    # FINAL DATAFRAMES
    # ========================================================

    fold_df = pd.DataFrame(
        fold_results
    )

    prediction_df = pd.DataFrame(
        prediction_rows
    )

    history_df = pd.concat(
        history_rows,
        ignore_index=True
    )

    # ========================================================
    # POOLED METRICS
    # ========================================================

    pooled_accuracy = accuracy_score(
        prediction_df["true_label"],
        prediction_df["pred_label"]
    )

    mean_accuracy = (
        fold_df["accuracy"].mean()
    )

    std_accuracy = (
        fold_df["accuracy"].std(
            ddof=0
        )
    )

    mean_balanced_accuracy = (
        fold_df[
            "balanced_accuracy"
        ].mean()
    )

    mean_kappa = (
        fold_df["kappa"].mean()
    )

    mean_auc = (
        fold_df["roc_auc"].mean()
    )

    # ========================================================
    # SAVE
    # ========================================================

    fold_df.to_csv(
        RESULTS_DIR /
        "optimized_fold_results.csv",
        index=False
    )

    prediction_df.to_csv(
        RESULTS_DIR /
        "optimized_predictions.csv",
        index=False
    )

    history_df.to_csv(
        RESULTS_DIR /
        "optimized_training_history.csv",
        index=False
    )

    # ========================================================
    # FINAL REPORT
    # ========================================================

    print()
    print("=" * 80)
    print(
        "FINAL OPTIMIZED BINARY MI RESULTS"
    )
    print("=" * 80)

    print(
        "Test subjects:",
        ", ".join(
            f"S{s:03d}"
            for s in TEST_SUBJECTS
        )
    )

    print()

    print(
        f"Pooled Accuracy       : "
        f"{pooled_accuracy*100:.2f}%"
    )

    print(
        f"Mean Fold Accuracy    : "
        f"{mean_accuracy*100:.2f}% "
        f"± {std_accuracy*100:.2f}%"
    )

    print(
        f"Mean Balanced Accuracy: "
        f"{mean_balanced_accuracy*100:.2f}%"
    )

    print(
        f"Mean Cohen's Kappa    : "
        f"{mean_kappa:.4f}"
    )

    print(
        f"Mean ROC-AUC          : "
        f"{mean_auc:.4f}"
    )

    print()

    print(
        fold_df[
            [
                "fold",
                "test_subject",
                "accuracy",
                "balanced_accuracy",
                "kappa",
                "roc_auc"
            ]
        ].to_string(
            index=False
        )
    )

    return (
        fold_df,
        prediction_df,
        history_df
    )

In [ ]:
# ============================================================
# CELL 18 — RUN OPTIMIZED EXPERIMENT
# ============================================================

fold_df, prediction_df, history_df = (
    run_optimized_binary_experiment()
)

## CELL 16 — Execution order and expected outputs

Run the notebook in order after restarting the kernel:

**01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12 → 13 → 14 → 15**

The final binary experiment uses:

- **Runs:** 4, 8, 12
- **Class 0:** Left Hand MI
- **Class 1:** Right Hand MI
- **Classes:** 2
- **Chance accuracy:** 50%
- **Test folds:** 10
- **Training subjects per fold:** up to 99

Saved results:

`results_s3_binary_da/fold_metrics.csv`

`results_s3_binary_da/test_predictions.csv`

`results_s3_binary_da/epoch_history.csv`

`results_s3_binary_da/test_embeddings.csv`

`results_s3_binary_da/summary.json`

Figures include binary fold accuracy, normalized confusion matrix, precision/recall/F1, ROC-AUC, training losses, architecture, and t-SNE.
